# Testing Notebook B: Pipeline and Scenario Validation

This notebook validates aligned, contradictory, and sparse scenario behavior across the worker -> arbiter -> master handoff.

It uses the primary notebook's synthetic scenario helpers and arbiter demo fixtures rather than pretending live providers already exist.

## 1. Load the Primary Notebook Namespace

In [ ]:
from __future__ import annotations

import contextlib
import io
import json
import runpy
import tempfile
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PRIMARY_NOTEBOOK = PROJECT_ROOT / "notebooks" / "01_biotech_disclosure_pipeline.ipynb"


def load_primary_notebook_namespace(notebook_path: Path) -> dict[str, Any]:
    payload = json.loads(notebook_path.read_text())
    namespace: dict[str, Any] = {"__name__": "__notebook__"}
    code_blocks = ["".join(cell.get("source", [])) for cell in payload["cells"] if cell["cell_type"] == "code"]
    notebook_code = "\n\n".join(code_blocks)
    with tempfile.NamedTemporaryFile("w", suffix="_primary_notebook.py", delete=False) as handle:
        handle.write(notebook_code)
        temp_path = Path(handle.name)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            loaded = runpy.run_path(str(temp_path), init_globals=namespace)
        namespace.update(loaded)
    finally:
        temp_path.unlink(missing_ok=True)
    return namespace


NS = load_primary_notebook_namespace(PRIMARY_NOTEBOOK)
EXPORTS = NS["NOTEBOOK_EXPORTS"]


def run_test(name: str, fn) -> None:
    try:
        fn()
        print(f"[PASS] {name}")
    except AssertionError as exc:
        print(f"[FAIL] {name}: {exc}")
        raise

## 2. Scenario Validation Tests

In [ ]:
DocumentType = EXPORTS["DocumentType"]
ProcessingStatus = EXPORTS["ProcessingStatus"]
SentimentLabel = EXPORTS["SentimentLabel"]
WorkerOutput = EXPORTS["WorkerOutput"]
CrossDocumentArbiter = EXPORTS["CrossDocumentArbiter"]
IntegratedMasterNode = EXPORTS["IntegratedMasterNode"]
PIPELINE_CONFIG = EXPORTS["PIPELINE_CONFIG"]
ARBITER_DEMO_CASES = EXPORTS["ARBITER_DEMO_CASES"]
DEMO_ARBITER_OUTPUTS = EXPORTS["DEMO_ARBITER_OUTPUTS"]


def build_master_payload_from_case(case_name: str):
    worker_outputs = [WorkerOutput.model_validate(output.model_dump(mode="python")) for output in ARBITER_DEMO_CASES[case_name]]
    arbiter_output = DEMO_ARBITER_OUTPUTS[case_name]
    master = IntegratedMasterNode()
    return master.build_payload(
        NS["MasterInput"](
            run_id=f"scenario_{case_name}",
            ticker="FAKE",
            worker_outputs=worker_outputs,
            arbiter_outputs=[arbiter_output],
            retrieval_results=[],
            config=PIPELINE_CONFIG,
        )
    )


def test_mostly_aligned_multi_document_case() -> None:
    arbiter_output = DEMO_ARBITER_OUTPUTS["mostly_aligned"]
    payload = build_master_payload_from_case("mostly_aligned")
    assert arbiter_output.aligned_signals, "aligned case should produce aligned signals"
    assert payload.overall_sentiment_label in {SentimentLabel.POSITIVE, SentimentLabel.MIXED}, "aligned case should not resolve as outright negative"
    assert payload.key_positive_signals, "aligned case should surface key positive signals"


def test_contradictory_cross_document_case() -> None:
    arbiter_output = DEMO_ARBITER_OUTPUTS["contradictory"]
    payload = build_master_payload_from_case("contradictory")
    assert arbiter_output.conflicting_signals, "contradictory case should surface explicit conflicts"
    assert payload.overall_sentiment_label == SentimentLabel.MIXED, f"expected mixed sentiment, got {payload.overall_sentiment_label}"
    assert payload.fogging_or_story_substance_flags, "contradictory case should preserve story-vs-substance flags"


def test_sparse_missing_document_case() -> None:
    arbiter_output = DEMO_ARBITER_OUTPUTS["sparse_missing"]
    payload = build_master_payload_from_case("sparse_missing")
    assert arbiter_output.missing_document_types, "sparse case should retain missing document types"
    assert payload.missing_document_types, "final payload should retain missing document types"
    assert payload.status == ProcessingStatus.PARTIAL, f"expected partial status, got {payload.status}"


def test_worker_arbiter_master_handoff_consistency() -> None:
    payload = build_master_payload_from_case("mostly_aligned")
    worker_types = {output.document_type for output in ARBITER_DEMO_CASES["mostly_aligned"]}
    payload_types = {card.document_type for card in payload.disclosures}
    assert worker_types == payload_types, "master disclosures should map one-for-one to worker outputs"


def test_preservation_of_uncertainty_and_conflict() -> None:
    contradictory = DEMO_ARBITER_OUTPUTS["contradictory"]
    sparse = DEMO_ARBITER_OUTPUTS["sparse_missing"]
    assert contradictory.conflicting_signals, "contradictory case should preserve conflicts"
    assert sparse.unresolved_uncertainties, "sparse case should preserve unresolved uncertainties"
    sparse_payload = build_master_payload_from_case("sparse_missing")
    assert sparse_payload.key_uncertainties, "sparse case should preserve uncertainty into final payload"


def test_story_vs_substance_mismatch_handling() -> None:
    payload = build_master_payload_from_case("contradictory")
    assert any("narrative" in flag.lower() or "story" in flag.lower() or "promotional" in flag.lower() or "polished" in flag.lower() for flag in payload.fogging_or_story_substance_flags), (
        "expected story-vs-substance or promotional mismatch flags"
    )


def test_overall_sentiment_remains_mixed_when_appropriate() -> None:
    payload = build_master_payload_from_case("contradictory")
    assert payload.overall_sentiment_label == SentimentLabel.MIXED, "contradictory case should remain mixed"


def test_low_confidence_handling() -> None:
    payload = build_master_payload_from_case("sparse_missing")
    assert payload.overall_confidence is None or payload.overall_confidence <= 0.5, f"expected low confidence, got {payload.overall_confidence}"


def test_final_payload_completeness_for_ui_use() -> None:
    payload = build_master_payload_from_case("mostly_aligned")
    assert payload.overall_summary, "final payload should include overall summary"
    assert payload.disclosures, "final payload should include disclosure cards"
    assert payload.provenance, "final payload should include provenance"
    for card in payload.disclosures:
        assert card.document_type is not None
        assert card.status is not None
        assert isinstance(card.warnings, list)
        assert isinstance(card.caveats, list)


TESTS = [
    ("mostly aligned multi-document case", test_mostly_aligned_multi_document_case),
    ("contradictory cross-document case", test_contradictory_cross_document_case),
    ("sparse / missing-document case", test_sparse_missing_document_case),
    ("worker -> arbiter -> master handoff consistency", test_worker_arbiter_master_handoff_consistency),
    ("preservation of uncertainty and conflict", test_preservation_of_uncertainty_and_conflict),
    ("story-vs-substance mismatch handling", test_story_vs_substance_mismatch_handling),
    ("overall sentiment remains mixed when appropriate", test_overall_sentiment_remains_mixed_when_appropriate),
    ("low-confidence handling", test_low_confidence_handling),
    ("final payload completeness for UI use", test_final_payload_completeness_for_ui_use),
]

for name, fn in TESTS:
    run_test(name, fn)

## 3. Completion Notes

- Covered categories: aligned, contradictory, and sparse scenarios; worker -> arbiter -> master consistency; preserved uncertainty and conflict; story-vs-substance handling; mixed-sentiment retention; low-confidence behavior; and final payload completeness.
- Future work: larger scenario libraries, real-source replay harnesses, and notebook-based regression automation.
- Unless otherwise noted, these tests use synthetic/mock scenario data from the primary notebook.